# ARC Prize 2026 — ARC-AGI-2

Deterministic, CPU-only, **Python standard library only**. No model, no
pretraining, no internet, no accelerator.

## What this is

A library of explicit transformation rules. A rule is applied to a task **only if
it reproduces every training demonstration exactly**, and it returns nothing
rather than guess when the rule is underdetermined for a given input.

This entry exists to test one claim:

> For a non-learning solver on ARC-AGI-2 the binding constraint is **hypothesis
> coverage, not hypothesis selection**. Demonstration-consistency is a
> near-perfect selector, so accuracy is bought by adding rules at no precision
> cost.

It is falsified if precision falls as the rule library grows. On the public
training split, growing the library from 6 to 21 rules moved accuracy from 0.80%
to 2.00% with precision@1 flat at 100% and zero ambiguity between fitting rules.

**Expect a low score, and read the writeup for why that is the result rather
than an excuse.** No rule in this library fits any task in the public evaluation
split, which is itself evidence about ARC-AGI-2's design.

## Reproducibility

Source: https://github.com/sufian07/arc-prize-2026

This notebook is generated from that repository by `kaggle/build_notebook.py`;
the code cells below are the verbatim contents of `arc_agi_2/solvers.py` and
`kaggle/kaggle_runner.py`. All code is open source, as prize eligibility
requires.


## `arc_agi_2/solvers.py`


In [ ]:
"""Deterministic baseline solvers for ARC-AGI-2.

Stdlib only, on purpose. A grid is a list of rows, each row a list of ints 0-9.

The point of this module is not to score well. It is to make the *shape* of a
solver explicit: a solver looks at a task's training pairs, decides whether it
can explain them, and if so predicts an output for a new input. Everything
smarter plugs into the same two functions, so the submission pipeline never
changes again.

Selection rule: a solver is only used on a task if it reproduces **every**
training output exactly. That is the one honest signal available before the
hidden test grid is scored, and it costs nothing to check.
"""

from collections import Counter

Grid = list  # list[list[int]]


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------

def dims(g: Grid) -> tuple:
    return len(g), len(g[0]) if g else 0


def same(a: Grid, b: Grid) -> bool:
    return a == b


def rot90(g: Grid) -> Grid:
    """Clockwise."""
    return [list(r) for r in zip(*g[::-1])]


def rot180(g: Grid) -> Grid:
    return [row[::-1] for row in g[::-1]]


def rot270(g: Grid) -> Grid:
    return [list(r) for r in zip(*g)][::-1]


def anti_transpose(g: Grid) -> Grid:
    return rot180([list(r) for r in zip(*g)])


def tile(g: Grid, a: int, b: int) -> Grid:
    wide = [row * b for row in g]
    return [r[:] for _ in range(a) for r in wide]


def scale(g: Grid, a: int, b: int) -> Grid:
    out = []
    for row in g:
        wide = [v for v in row for _ in range(b)]
        out.extend(wide[:] for _ in range(a))
    return out


def background(g: Grid) -> int:
    """The most common colour. Ties break to the lower colour, deterministically."""
    counts = Counter(v for row in g for v in row)
    return min(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0]


# ---------------------------------------------------------------------------
# solvers
#
# Each is a callable: fit(train_pairs) -> predict(input_grid) -> Grid | None.
# fit returns None when the solver cannot explain the training pairs at all,
# which is cheaper than predicting and checking afterwards.
# ---------------------------------------------------------------------------

def fit_identity(pairs):
    """Output is the input, untouched."""
    if all(same(p["input"], p["output"]) for p in pairs):
        return lambda g: [row[:] for row in g]
    return None


def fit_constant(pairs):
    """Every training pair produces the same output, whatever the input."""
    outs = [p["output"] for p in pairs]
    first = outs[0]
    if all(same(o, first) for o in outs):
        return lambda g, out=first: [row[:] for row in out]
    return None


def fit_colour_map(pairs):
    """Same shape in and out, with one consistent colour substitution.

    Rejects any mapping that is ambiguous — a colour sending to two different
    colours across the training set means this is not what the task is doing,
    and guessing which one wins would be inventing a rule.
    """
    mapping: dict = {}
    for p in pairs:
        gi, go = p["input"], p["output"]
        if dims(gi) != dims(go):
            return None
        for ri, ro in zip(gi, go):
            for a, b in zip(ri, ro):
                if mapping.setdefault(a, b) != b:
                    return None

    def predict(g, m=mapping):
        # Refuse rather than guess. A colour absent from every training input has
        # no learned image, and passing it through unchanged is a guess dressed
        # as a rule. Training task aabf363d is the case: colour 8 appears only at
        # test time, the identity fallthrough got 16 of 49 cells wrong, and the
        # task counted as solved-then-failed instead of honestly abstained.
        # Same discipline as rejecting a rule that misses one demonstration cell.
        if any(v not in m for row in g for v in row):
            return None
        return [[m[v] for v in row] for row in g]

    return predict


def fit_transpose(pairs):
    for p in pairs:
        gi, go = p["input"], p["output"]
        if [list(r) for r in zip(*gi)] != go:
            return None
    return lambda g: [list(r) for r in zip(*g)]


def fit_flip_h(pairs):
    for p in pairs:
        if [row[::-1] for row in p["input"]] != p["output"]:
            return None
    return lambda g: [row[::-1] for row in g]


def fit_flip_v(pairs):
    for p in pairs:
        if p["input"][::-1] != p["output"]:
            return None
    return lambda g: g[::-1]


def fit_most_common_colour(pairs):
    """Degenerate fallback: a 1x1 grid of the input's most common colour.

    Never fits a real task. It exists so `attempt_2` is always populated with
    something well-formed rather than a duplicate of attempt_1.
    """
    return None


# ---------------------------------------------------------------------------
# tranche 2 - added 2026-09-05
#
# These exist to move one number: the size of the hypothesis library. The claim
# under test (see ../../contests/CONTESTS.md) is that demonstration-consistency
# stays a near-perfect selector as the library grows. That is only measurable
# if the library actually grows, in tranches, with precision recorded at each
# step. See `measure.py`.
#
# Every one obeys the same two rules as tranche 1: reject unless the rule
# reproduces *every* demonstration exactly, and return None rather than guess
# when the rule is underdetermined for this particular input.
# ---------------------------------------------------------------------------

def _whole_grid(op):
    """Build a fit function for a parameterless whole-grid geometric op."""

    def fit(pairs, op=op):
        for p in pairs:
            if op(p["input"]) != p["output"]:
                return None
        return lambda g, op=op: op(g)

    return fit


fit_rot90 = _whole_grid(rot90)
fit_rot180 = _whole_grid(rot180)
fit_rot270 = _whole_grid(rot270)
fit_anti_transpose = _whole_grid(anti_transpose)
fit_mirror_h = _whole_grid(lambda g: [row + row[::-1] for row in g])
fit_mirror_v = _whole_grid(lambda g: [r[:] for r in g] + [r[:] for r in g[::-1]])


def fit_tile(pairs):
    """Output is the input repeated a x b times. One factor pair for the task."""
    factors = set()
    for p in pairs:
        gi, go = p["input"], p["output"]
        h, w = dims(gi)
        H, W = dims(go)
        if not h or not w or H % h or W % w:
            return None
        a, b = H // h, W // w
        if (a, b) == (1, 1):
            return None  # identity wearing a tiling costume
        if tile(gi, a, b) != go:
            return None
        factors.add((a, b))
    if len(factors) != 1:
        return None
    (a, b), = factors
    return lambda g, a=a, b=b: tile(g, a, b)


def fit_scale(pairs):
    """Each cell becomes an a x b block. One factor pair for the task."""
    factors = set()
    for p in pairs:
        gi, go = p["input"], p["output"]
        h, w = dims(gi)
        H, W = dims(go)
        if not h or not w or H % h or W % w:
            return None
        a, b = H // h, W // w
        if (a, b) == (1, 1):
            return None
        if scale(gi, a, b) != go:
            return None
        factors.add((a, b))
    if len(factors) != 1:
        return None
    (a, b), = factors
    return lambda g, a=a, b=b: scale(g, a, b)


def _unscale(g: Grid, a: int, b: int):
    """Inverse of scale, or None when a block is not one colour."""
    h, w = dims(g)
    if not h or not w or h % a or w % b:
        return None
    out = []
    for i in range(0, h, a):
        row = []
        for j in range(0, w, b):
            block = {g[i + di][j + dj] for di in range(a) for dj in range(b)}
            if len(block) != 1:
                return None
            row.append(block.pop())
        out.append(row)
    return out


def fit_downscale(pairs):
    """Block-reduce by a x b, where every block is a single colour."""
    factors = set()
    for p in pairs:
        gi, go = p["input"], p["output"]
        h, w = dims(gi)
        H, W = dims(go)
        if not H or not W or h % H or w % W:
            return None
        a, b = h // H, w // W
        if (a, b) == (1, 1):
            return None
        if _unscale(gi, a, b) != go:
            return None
        factors.add((a, b))
    if len(factors) != 1:
        return None
    (a, b), = factors
    # A non-uniform block at test time means the rule does not apply here, so
    # _unscale returns None. That is an abstention, not a guess.
    return lambda g, a=a, b=b: _unscale(g, a, b)


def _bbox(g: Grid, bg: int):
    rows = [i for i, row in enumerate(g) if any(v != bg for v in row)]
    if not rows:
        return None
    cols = [j for j in range(len(g[0])) if any(row[j] != bg for row in g)]
    if not cols:
        return None
    return [row[min(cols):max(cols) + 1] for row in g[min(rows):max(rows) + 1]]


def fit_crop_bbox(pairs):
    """Crop to the bounding box of everything that is not the background."""
    for p in pairs:
        gi = p["input"]
        if _bbox(gi, background(gi)) != p["output"]:
            return None
    return lambda g: _bbox(g, background(g))


def fit_strip_border(pairs):
    """Drop one cell from every edge."""
    for p in pairs:
        gi = p["input"]
        h, w = dims(gi)
        if h < 3 or w < 3:
            return None
        if [row[1:-1] for row in gi[1:-1]] != p["output"]:
            return None
    return lambda g: ([row[1:-1] for row in g[1:-1]]
                      if len(g) >= 3 and len(g[0]) >= 3 else None)


def fit_add_border(pairs):
    """Surround the input with a one-cell frame of a single fixed colour."""
    colours = set()
    for p in pairs:
        gi, go = p["input"], p["output"]
        h, w = dims(gi)
        if dims(go) != (h + 2, w + 2):
            return None
        frame = set(go[0]) | set(go[-1]) | {r[0] for r in go} | {r[-1] for r in go}
        if len(frame) != 1:
            return None
        if [row[1:-1] for row in go[1:-1]] != gi:
            return None
        colours.add(frame.pop())
    if len(colours) != 1:
        return None
    c, = colours
    return lambda g, c=c: ([[c] * (len(g[0]) + 2)]
                           + [[c] + row[:] + [c] for row in g]
                           + [[c] * (len(g[0]) + 2)])


def _pick(sel):
    """1x1 output holding a colour chosen from the input by `sel`."""

    def fit(pairs, sel=sel):
        for p in pairs:
            c = sel(p["input"])
            if c is None or p["output"] != [[c]]:
                return None
        return lambda g, sel=sel: (None if sel(g) is None else [[sel(g)]])

    return fit


def _most_common(g):
    counts = Counter(v for row in g for v in row)
    return min(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0] if counts else None


def _least_common(g):
    counts = Counter(v for row in g for v in row)
    return min(counts.items(), key=lambda kv: (kv[1], kv[0]))[0] if counts else None


def _most_common_non_bg(g):
    counts = Counter(v for row in g for v in row)
    if len(counts) < 2:
        return None
    del counts[background(g)]
    return min(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0]


fit_pick_most_common = _pick(_most_common)
fit_pick_least_common = _pick(_least_common)
fit_pick_most_common_non_bg = _pick(_most_common_non_bg)


# Order matters: the first fitting solver becomes attempt_1.
SOLVERS = [
    ("constant", fit_constant),
    ("identity", fit_identity),
    ("colour_map", fit_colour_map),
    ("transpose", fit_transpose),
    ("flip_h", fit_flip_h),
    ("flip_v", fit_flip_v),
    ("rot90", fit_rot90),
    ("rot180", fit_rot180),
    ("rot270", fit_rot270),
    ("anti_transpose", fit_anti_transpose),
    ("mirror_h", fit_mirror_h),
    ("mirror_v", fit_mirror_v),
    ("tile", fit_tile),
    ("scale", fit_scale),
    ("downscale", fit_downscale),
    ("crop_bbox", fit_crop_bbox),
    ("strip_border", fit_strip_border),
    ("add_border", fit_add_border),
    ("pick_most_common", fit_pick_most_common),
    ("pick_least_common", fit_pick_least_common),
    ("pick_most_common_non_bg", fit_pick_most_common_non_bg),
]

# Which tranche each solver joined in. The coverage-vs-selection claim is tested
# by replaying the library one tranche at a time and recording precision at each
# step, so this is data the experiment reads, not commit history. See measure.py.
TRANCHE = {name: (1 if i < 6 else 2) for i, (name, _) in enumerate(SOLVERS)}


# ---------------------------------------------------------------------------
# prediction
# ---------------------------------------------------------------------------

def fallback(pairs, grid: Grid) -> Grid:
    """Something well-formed when nothing fits.

    Copies the input. Wrong almost always, but it is a valid grid, and a valid
    wrong answer keeps the submission parseable — a malformed one can invalidate
    the whole file.
    """
    return [row[:] for row in grid]


def second_guess(pairs, grid: Grid) -> Grid:
    """A different-but-plausible answer for attempt_2.

    Uses the most common training output shape filled with that output's most
    common colour. Distinct from the identity fallback, so the two attempts are
    never the same grid.
    """
    out = pairs[0]["output"]
    h, w = dims(out)
    counts = Counter(v for row in out for v in row)
    fill = counts.most_common(1)[0][0] if counts else 0
    return [[fill] * w for _ in range(h)]


def predict(task) -> list:
    """Two attempts for every test input in a task.

    Returns a list of {"attempt_1": grid, "attempt_2": grid}, one per test
    input, which is exactly the per-task value the submission file wants.
    """
    pairs = task["train"]
    fitted = []
    for name, fit in SOLVERS:
        fn = fit(pairs)
        if fn is not None:
            fitted.append((name, fn))

    out = []
    for test in task["test"]:
        g = test["input"]
        # A fitted solver may still abstain on this particular input, so collect
        # what actually produced a grid rather than trusting the fit alone.
        got = [p for _, fn in fitted if (p := fn(g)) is not None]
        a1 = got[0] if got else fallback(pairs, g)
        a2 = got[1] if len(got) > 1 else second_guess(pairs, g)
        if a1 == a2:
            a2 = second_guess(pairs, g)
        out.append({"attempt_1": a1, "attempt_2": a2})
    return out


def fitted_names(task) -> list:
    """Which solvers explain this task's training pairs. For reporting."""
    return [n for n, fit in SOLVERS if fit(task["train"]) is not None]


## `kaggle/kaggle_runner.py`


In [ ]:
"""The Kaggle side of an ARC-AGI-2 submission: find the tasks, answer, write the file.

Runs with **no internet**, so nothing here fetches, installs or clones. It is
stdlib only for the same reason the solvers are.

The hard part is not prediction, it is not guessing wrong about the input
layout. A submission that predicts for the wrong set of task ids scores zero no
matter how good the rules are, and the competition directory is not visible from
here — the 2026 layout is only observable from inside a Kaggle session. So this
discovers the layout rather than assuming it, and handles every shape ARC Prize
has shipped:

  per-task files       one <task_id>.json per task, as the public repo ships
  consolidated files   arc-agi_test_challenges.json, a dict of task_id -> task
  a sample submission  sample_submission.json, whose keys are authoritative

**`sample_submission.json` wins when present.** It states exactly which task ids
the grader expects and how many attempt slots each needs, which is better
evidence than any filename convention. Everything else is a fallback for when it
is absent.

Every discovery decision is printed. A submission that fails at 3am on the
deadline is diagnosed from the notebook log or not at all.

Local test against the public evaluation set:

    python kaggle/kaggle_runner.py --input data/ARC-AGI-2/data/evaluation --out /tmp/s.json
"""

import argparse
import json
import sys
from pathlib import Path

try:  # running from the repo: make the repo root importable, as run.py does
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:  # pragma: no cover - no __file__ inside a notebook cell
    pass

try:  # locally this resolves; inside the notebook the cell above defined it
    from arc_agi_2.solvers import predict
except ImportError:  # pragma: no cover - notebook path
    predict = globals()["predict"]


# ---------------------------------------------------------------------------
# discovery
# ---------------------------------------------------------------------------

def _is_task(o) -> bool:
    return isinstance(o, dict) and "train" in o and "test" in o


def _is_sample_submission(o) -> bool:
    if not isinstance(o, dict) or not o:
        return False
    for v in o.values():
        if not isinstance(v, list) or not v:
            return False
        if not all(isinstance(a, dict) and "attempt_1" in a for a in v):
            return False
    return True


def discover(root: Path) -> tuple:
    """Return (ids, tasks, sample) for whatever layout `root` turns out to hold."""
    pool: dict = {}
    consolidated: dict = {}
    sample = None
    sample_path = None

    files = sorted(p for p in root.rglob("*.json"))
    print(f"scanning {root}: {len(files)} json file(s)")

    for p in files:
        try:
            o = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"  skip {p.name}: unreadable ({type(e).__name__})")
            continue
        if _is_task(o):
            pool[p.stem] = o
        elif _is_sample_submission(o):
            sample, sample_path = o, p
            print(f"  {p.name}: sample submission, {len(o)} task ids")
        elif isinstance(o, dict) and o and all(_is_task(v) for v in o.values()):
            consolidated[p] = o
            print(f"  {p.name}: {len(o)} tasks")
        # anything else (solutions files, metadata) is deliberately ignored

    per_file = len(pool)
    if per_file:
        print(f"  {per_file} per-task file(s)")

    # Consolidated files merge into the pool. Sorted so a "test"-named file is
    # applied last and wins any id collision, which is the set being graded.
    for p in sorted(consolidated, key=lambda q: "test" in q.name.lower()):
        pool.update(consolidated[p])

    if sample is not None:
        ids = list(sample)
        print(f"required ids: {len(ids)}, from {sample_path.name}")
    else:
        test_files = [p for p in consolidated if "test" in p.name.lower()]
        if test_files:
            pick = sorted(test_files, key=lambda q: len(q.name))[0]
            ids = list(consolidated[pick])
            print(f"no sample submission; using {pick.name} ({len(ids)} ids)")
        else:
            ids = sorted(pool)
            print(f"no sample submission and no test file; using all {len(ids)} tasks")

    return ids, pool, sample


# ---------------------------------------------------------------------------
# validation
# ---------------------------------------------------------------------------

def validate(submission: dict, ids: list, tasks: dict, sample) -> list:
    """Structural checks. A malformed file can void the whole submission."""
    problems = []
    for tid in ids:
        entry = submission.get(tid)
        if entry is None:
            problems.append(f"{tid}: missing")
            continue
        want = len(tasks[tid]["test"]) if tid in tasks else None
        if sample is not None and len(sample.get(tid, [])) != len(entry):
            problems.append(
                f"{tid}: {len(entry)} attempt block(s), sample wants {len(sample[tid])}"
            )
        elif want is not None and len(entry) != want:
            problems.append(f"{tid}: {len(entry)} attempt block(s) for {want} test input(s)")
        for i, att in enumerate(entry):
            for key in ("attempt_1", "attempt_2"):
                g = att.get(key)
                if not isinstance(g, list) or not g:
                    problems.append(f"{tid}[{i}].{key}: not a non-empty grid")
                elif not all(isinstance(r, list) and r for r in g):
                    problems.append(f"{tid}[{i}].{key}: rows must be non-empty lists")
                elif len({len(r) for r in g}) != 1:
                    problems.append(f"{tid}[{i}].{key}: ragged rows")
                elif not all(isinstance(v, int) and 0 <= v <= 9 for r in g for v in r):
                    problems.append(f"{tid}[{i}].{key}: values must be ints 0-9")
    extra = set(submission) - set(ids)
    if extra:
        problems.append(f"{len(extra)} unrequested id(s): {sorted(extra)[:5]}")
    return problems


# ---------------------------------------------------------------------------

def build(root: Path, out: Path) -> int:
    ids, tasks, sample = discover(root)

    missing = [t for t in ids if t not in tasks]
    if missing:
        print(f"\nFATAL: {len(missing)} required id(s) have no task data: {missing[:5]}")
        return 1
    if not ids:
        print("\nFATAL: no tasks discovered. Check the attached competition data.")
        return 1

    submission = {tid: predict(tasks[tid]) for tid in ids}

    problems = validate(submission, ids, tasks, sample)
    if problems:
        print(f"\nINVALID — {len(problems)} problem(s), nothing written:")
        for p in problems[:20]:
            print("  " + p)
        return 1

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(submission), encoding="utf-8")
    kb = out.stat().st_size / 1024

    # How many answers came from a rule rather than the fallback. It does not
    # change the file, but a run that answers nothing by rule should say so
    # rather than look identical to one that did.
    try:
        from arc_agi_2.solvers import fitted_names
    except ImportError:  # pragma: no cover - notebook path
        fitted_names = globals().get("fitted_names")
    if fitted_names:
        n = sum(1 for tid in ids if fitted_names(tasks[tid]))
        print(f"\nrules fit {n}/{len(ids)} task(s); the rest are the fallback")

    print(f"wrote {out} — {len(submission)} tasks, {kb:,.0f} KB, format OK")
    return 0


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--input", type=Path, default=Path("/kaggle/input"))
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/submission.json"))
    a = ap.parse_args(argv)
    if not a.input.exists():
        print(f"FATAL: {a.input} does not exist")
        return 1
    return build(a.input, a.out)


## Run


In [ ]:
# Build the submission. Kaggle's own paths, no arguments to get wrong.
rc = build(Path("/kaggle/input"), Path("/kaggle/working/submission.json"))
assert rc == 0, "submission was NOT written — read the log above before submitting"
